In [1]:
import os, torch
print("working folder:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

working folder: /content
GPU: Tesla T4


In [2]:
%cd /content
!git clone https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git repo
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin

/content
fatal: destination path 'repo' already exists and is not an empty directory.
/content/repo/pinyin_scorer


In [3]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/pinyin_runs

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
stage1	stage2	stage3	stage4


In [4]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo log --oneline -1
!ls *.py
!grep -c "Hiss only" data.py

/content/repo/pinyin_scorer
877929f (HEAD -> main, origin/main, origin/HEAD) training a new model
data.py		 model.py	     prepare_folders.py    train.py
diagnose.py	 pinyin_units.py     prepare_syllables.py
eval_sources.py  prepare_aishell.py  score.py
1


In [3]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/processed.zip -d /content/ext
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/ext/processed --out-dir manifests/folders

train: 6560 clips -> manifests/syl_train.jsonl
test: 3279 clips -> manifests/syl_test.jsonl
clips per speaker: {'FV1': 1640, 'FV2': 1640, 'FV3': 1639, 'MV1': 1640, 'MV2': 1640, 'MV3': 1640}
davinfifield     train   2485 clips
digmandarin      train   3254 clips
gitcode          test    1283 clips
hotoo            train   3232 clips
hugolpz          train   3312 clips  (8 skipped)
losttheory       train   3136 clips
mandarintemple   train   3206 clips
misc             train    336 clips
shikangkai_f     test    1615 clips
shikangkai_m     train   1425 clips
sreading         train   3232 clips
studycli         test    3144 clips
uvm              test    1300 clips
yabla            train   4884 clips
yoyochinese      train   4836 clips

train: 33338 clips (7.6 h)   test: 7342 clips


In [ ]:
!grep -c "Hiss only" data.py
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

1
Loading weights: 100% 211/211 [00:09<00:00, 21.43it/s]
39898 training clips, 8.9 hours, 626 batches per epoch, device=cuda, amp=True
before training initial_error=0.0360  final_error=0.0372  tone_error=0.0128  mean_error=0.0287  syllable_initial_acc=0.9643  syllable_final_acc=0.9637  syllable_tone_acc=0.9884  syllable_all3_acc=0.9283  syllable_clips=3279
/content/repo/pinyin_scorer/train.py:191: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch 1 batch 50/626 step 50 initial=1.349 final=0.910 tone=0.280 lr=2.55e-06
epoch 1 batch 100/626 step 100 initial=1.062 final=0.866 tone=0.197 lr=5.05e-06
epoch 1 batch 150/626 step 150 i

In [5]:
!ps aux | grep "train.py" | grep -v grep | wc -l
!ls /content/repo/pinyin_scorer/manifests /content/ext 2>&1 | head -5
!ls /content/drive/MyDrive/pinyin_runs/stage4/last

3
/content/ext:
__MACOSX
processed

/content/repo/pinyin_scorer/manifests:
encoder  top.pt  units.json


In [6]:
%cd /content/repo/pinyin_scorer

/content/repo/pinyin_scorer


In [7]:
!setsid nohup python -u train.py --init-from /content/drive/MyDrive/pinyin_runs/stage4/last \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4b \
    --epochs 5 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 100 --augment --workers 2 \
    > /content/train_stage4b.log 2>&1 &

In [8]:
!tail -3 /content/train_stage4b.log
!grep "^epoch [0-9]* initial" /content/train_stage4b.log

In [9]:
!python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/stage4b/best \
    --manifests manifests/syl_test.jsonl manifests/folders/test_*.jsonl --out stage4_heldout.csv

Traceback (most recent call last):
  File "/content/repo/pinyin_scorer/eval_sources.py", line 19, in <module>
    from score import Scorer
  File "/content/repo/pinyin_scorer/score.py", line 22, in <module>
    from model import PinyinCTC
  File "/content/repo/pinyin_scorer/model.py", line 14, in <module>
    from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
  File "/usr/local/lib/python3.13/dist-packages/transformers/__init__.py", line 847, in <module>
    for _proc_file in sorted((Path(__file__).parent / "models").rglob("image_processing_*.py")):
                      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/glob.py", line 505, in select_recursive
    yield from select_recursive_step(stack, match_pos)
  File "/usr/lib/python3.13/glob.py", line 529, in select_recursive_step
    yield from select_next(entry_path, exists=True)
  File "/usr/lib/python3.13/glob.py", line 456, in select_wildcard
    entries = list(

In [11]:
!ps aux | grep "train.py" | grep -v grep | grep -o "\-\-out [^ ]*" | sort -u

--out /content/drive/MyDrive/pinyin_runs/stage4


In [15]:
import json
for line in open("/content/drive/MyDrive/pinyin_runs/stage4/log.jsonl"):
    d = json.loads(line)
    print(d["epoch"], *(f"{d[k]:.3f}" for k in
          ["syllable_initial_acc", "syllable_final_acc", "syllable_tone_acc", "syllable_all3_acc"]))

1 0.972 0.894 0.942 0.842
2 0.967 0.937 0.939 0.866
3 0.950 0.865 0.892 0.780
4 0.969 0.914 0.935 0.852
5 0.969 0.864 0.846 0.755


In [17]:
!pkill -f train.py
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "no-augment-sources" prepare_folders.py; grep -c "duplicate_pairs" find_duplicates.py; grep -c "def speech_power" data.py

/content/repo/pinyin_scorer
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 3.54 KiB | 1.77 MiB/s, done.
From https://github.com/samyuktaathreya/pinyin_audio_recognizer_model
   a5c183c..5ed724b  main       -> origin/main
Updating a5c183c..5ed724b
Fast-forward
 pinyin_scorer/find_duplicates.py | 113 +++++++++++++++++++++++++++++++++++++++
 train.ipynb                      |  95 ++++++++++++++++++++++++++++++++
 2 files changed, 208 insertions(+)
 create mode 100644 pinyin_scorer/find_duplicates.py
1
0
1


In [18]:
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std
!ls /content/std/standard

davinfifield  hotoo	    losttheory	    sreading  yabla
digmandarin   hugolpz	    mandarintemple  studycli  youtube_man
gitcode       left_out.csv  misc	    uvm       yoyochinese


In [19]:
ROOT = "/content/std/standard"
!python find_duplicates.py {ROOT}/*/

read  1617 syllables from davinfifield
read  1627 syllables from digmandarin
read  1283 syllables from gitcode
read  1616 syllables from hotoo
read  1660 syllables from hugolpz
read  1568 syllables from losttheory
read  1603 syllables from mandarintemple
read   168 syllables from misc
read  1616 syllables from sreading
read  1572 syllables from studycli
read  1300 syllables from uvm
read  1628 syllables from yabla
read     0 syllables from youtube_man
read  1612 syllables from yoyochinese

source A         source B          shared  same length  same recording?
gitcode          sreading            1283          93%              90%   <-- check
davinfifield     sreading            1594          17%              17%
losttheory       uvm                 1244          14%              11%
misc             sreading             165          10%               2%
sreading         yabla               1608          15%               1%
misc             yoyochinese          168          11%       

In [20]:
import pandas as pd
from IPython.display import Audio, display
ex = pd.read_csv("duplicate_examples.csv")
pair = ex[(ex.source_a == "losttheory") & (ex.source_b == "uvm")]
for _, r in pair.sample(3, random_state=0).iterrows():
    print(f"\n{r.syllable}  (similarity {r.similarity})")
    display(Audio(r.file_a)); display(Audio(r.file_b))


dao1  (similarity 0.986)



ang4  (similarity 0.978)



weng1  (similarity 0.976)


In [21]:
ROOT = "/content/std/standard"
!python prepare_folders.py --root {ROOT} --out-dir manifests/std \
    --test-sources gitcode studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory
!ls manifests/syl_train.jsonl manifests/syl_test.jsonl

davinfifield     train   1617 clips
digmandarin      train   1627 clips
gitcode          test    1283 clips
hotoo            train   1616 clips
hugolpz          train   1656 clips  (4 skipped)
mandarintemple   train   1603 clips
misc             train    168 clips
studycli         test    1572 clips
uvm              test    1300 clips  (no added noise)
yabla            train   1628 clips
yoyochinese      train   1612 clips

train: 11527 clips (3.1 h)   test: 4155 clips
manifests/syl_test.jsonl  manifests/syl_train.jsonl


In [22]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl

train: 6560 clips -> manifests/syl_train.jsonl
test: 3279 clips -> manifests/syl_test.jsonl
clips per speaker: {'FV1': 1640, 'FV2': 1640, 'FV3': 1639, 'MV1': 1640, 'MV2': 1640, 'MV3': 1640}


In [23]:
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/std/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage5 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

Loading weights: 100% 211/211 [00:00<00:00, 685.70it/s]
18087 training clips, 4.4 hours, 285 batches per epoch, device=cuda, amp=True
before training initial_error=0.0360  final_error=0.0372  tone_error=0.0128  mean_error=0.0287  syllable_initial_acc=0.9643  syllable_final_acc=0.9637  syllable_tone_acc=0.9884  syllable_all3_acc=0.9283  syllable_clips=3279
epoch 1 batch 50/285 step 50 initial=0.854 final=0.645 tone=0.219 lr=4.47e-06
epoch 1 batch 100/285 step 100 initial=0.659 final=0.515 tone=0.140 lr=8.86e-06
epoch 1 batch 150/285 step 150 initial=0.634 final=0.500 tone=0.161 lr=1.32e-05
epoch 1 batch 200/285 step 200 initial=0.649 final=0.640 tone=0.222 lr=1.50e-05
epoch 1 batch 250/285 step 250 initial=0.580 final=0.546 tone=0.152 lr=1.50e-05
epoch 1 initial_error=0.0381  final_error=0.0875  tone_error=0.0342  mean_error=0.0533  syllable_initial_acc=0.9622  syllable_final_acc=0.9125  syllable_tone_acc=0.9661  syllable_all3_acc=0.8634  syllable_clips=3279
Writing model shards: 100% 1

In [24]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_std.csv
!cp stage2_std.csv stage5_std.csv /content/drive/MyDrive/


######## stage2
Loading weights: 100% 211/211 [00:00<00:00, 651.82it/s]
source            clips  initial    final     tone    all 3
syl_test           3279    96.9%    96.5%    99.0%    93.1%
test_gitcode       1283    89.2%    76.6%    97.7%    67.7%
test_studycli      1572    96.5%    91.5%    85.2%    75.6%
test_uvm           1300    96.5%    92.9%    95.5%    86.8%
ALL                7434    95.4%    91.4%    95.2%    83.9%

most common mistakes (target -> heard, count):
  initial: q->x (26), d->b (16), ch->c (14), l->n (14), q->j (13), t->p (13), p->h (11), h->p (10), n->m (10), b->(none) (9)
  final  : eng->ang (79), ou->ao (46), u->ou (23), iu->iao (23), en->an (18), ang->a (16), i->ü (15), uo->ua (15), ing->in (14), un->uan (14)
  tone   : 2->3 (232), 3->2 (59), 3->4 (23), 1->2 (15), 2->1 (11), 4->3 (8), 3->1 (6)

1195 failures written to stage2_std.csv

######## stage5
Loading weights: 100% 211/211 [00:00<00:00, 816.53it/s]
source            clips  initial    final     tone  

In [25]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

18087 training clips: {'FV1': 1640, 'FV2': 1640, 'MV1': 1640, 'MV2': 1640, 'davinfifield': 1617, 'digmandarin': 1627, 'hotoo': 1616, 'hugolpz': 1656, 'mandarintemple': 1603, 'misc': 168, 'yabla': 1628, 'yoyochinese': 1612}
manifests/syl_test.jsonl: 3279 clips, speakers ['FV3', 'MV3'], speakers also in training: none, identical files in training: 0
manifests/std/test_gitcode.jsonl: 1283 clips, speakers ['gitcode'], speakers also in training: none, identical files in training: 0
manifests/std/test_studycli.jsonl: 1572 clips, speakers ['studycli'], speakers also in training: none, identical files in training: 0
manifests/std/test_uvm.jsonl: 1300 clips, speakers ['uvm'], speakers also in training: none, identical files in training: 0


In [26]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

usage: find_duplicates.py [-h] [--max-frames-apart MAX_FRAMES_APART]
                          [--min-similarity MIN_SIMILARITY] [--out OUT]
                          folders [folders ...]
find_duplicates.py: error: unrecognized arguments: --summary tp_dup_pairs.csv


In [27]:
import json, glob, random
from IPython.display import Audio, display
from score import Scorer
scorer = Scorer("/content/drive/MyDrive/pinyin_runs/stage5/best")
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    scored = []
    for r in random.Random(0).sample(rows, min(40, len(rows))):
        res = scorer.score(r["audio"], r["pinyin"])
        scored.append((min(p["score"] for p in res["parts"].values()), r, res["parts"]))
    scored.sort(key=lambda x: x[0])
    print(f"\n===== {tf}")
    for label, group in [("WORST", scored[:3]), ("BEST", scored[-3:])]:
        for conf, r, p in group:
            print(f"{label}: target {r['pinyin']:7} heard {p['initial']['heard']} + {p['final']['heard']} "
                  f"+ tone {p['tone']['heard']}   (weakest part {conf:.2f})")
            display(Audio(r["audio"]))

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]


===== manifests/syl_test.jsonl
WORST: target ta3     heard k + a + tone 3   (weakest part 0.00)


WORST: target diu3    heard d + ing + tone 3   (weakest part 0.04)


WORST: target diu2    heard l + iu + tone 2   (weakest part 0.10)


BEST: target cu3     heard c + u + tone 3   (weakest part 1.00)


...

WORST: target chui4   heard sh + ui + tone 4   (weakest part 0.12)


WORST: target kun3    heard k + un + tone 2   (weakest part 0.26)


BEST: target pin2    heard p + in + tone 2   (weakest part 1.00)


BEST: target liang2  heard l + iang + tone 2   (weakest part 1.00)


BEST: target mei4    heard m + ei + tone 4   (weakest part 1.00)
